In [1]:
!pip install transformers torch


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json
import requests

In [52]:
with open("corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print("Corpus size:", len(corpus))

Corpus size: 20902


In [53]:
nli_model = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli"
)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 515/515 [00:00<00:00, 3856.59it/s]


In [36]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4057.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
corpus_embeddings = model.encode(corpus, show_progress_bar=True)

Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 653/653 [01:32<00:00,  7.05it/s]


In [39]:
dimension = corpus_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(corpus_embeddings))

print("FAISS index built:", index.ntotal)

FAISS index built: 20887


In [40]:
faiss.write_index(index, "faiss_index.index")

In [41]:
def verify_claim(claim, evidence):

    result = nli_model(
        f"{claim} </s></s> {evidence}"
    )[0]

    label = result['label']
    score = result['score']

    return label, round(score, 3)

In [42]:
def verify_with_evidence(claim, evidences):

    results = []

    for e in evidences:
        label, score = verify_claim(claim, e)
        results.append((e, label, score))

    return results

In [43]:
def final_decision(results):

    support = 0
    contradict = 0
    neutral = 0

    for _, label, _ in results:
        if label == "ENTAILMENT":
            support += 1
        elif label == "CONTRADICTION":
            contradict += 1
        else:
            neutral += 1

    if support > contradict and support > neutral:
        return "SUPPORTED"
    elif contradict > support and contradict > neutral:
        return "CONTRADICTED"
    else:
        return "NEUTRAL"

In [44]:
from serpapi import GoogleSearch

def web_search_serpapi(query, max_results=5):

    params = {
        "q": query,
        "api_key": "a1998f816e6a8c676e95c504b8b0620ecfde0eab89b4fff491169872753f4093",
        "num": max_results
    }

    search = GoogleSearch(params)
    results = search.get_dict()

    snippets = []

    for r in results.get("organic_results", []):
        if "snippet" in r:
            snippets.append(r["snippet"])

    return snippets[:max_results]

In [45]:
def clean_web_results(results):

    cleaned = []

    for r in results:
        if len(r.split()) < 5:
            continue
        cleaned.append(r.strip())

    return list(set(cleaned))

In [46]:
import json

def add_to_corpus(new_texts):

    global corpus, index

    # 🔥 remove duplicates
    new_texts = [t for t in new_texts if t not in corpus]

    if not new_texts:
        print("⚠️ No unique data to add")
        return

    before = len(corpus)

    corpus.extend(new_texts)

    # 🔥 UPDATE FAISS
    new_embeddings = model.encode(new_texts)
    index.add(np.array(new_embeddings))

    after = len(corpus)

    print(f"Added {after - before} new entries")
    print(f"Corpus size: {before} → {after}")
    print(f"FAISS size: {index.ntotal}")

    # 🔥 SAVE TO FILE (IMPORTANT)
    with open("corpus.json", "w", encoding="utf-8") as f:
        json.dump(corpus, f, ensure_ascii=False, indent=2)

    print("💾 Corpus saved to file")

In [47]:
def hybrid_retrieve(claim, top_k=5, threshold=0.65, use_mock=False):

    claim_embedding = model.encode([claim])

    distances, indices = index.search(np.array(claim_embedding), top_k)

    similarity = 1 / (1 + distances[0][0])

    local_results = [corpus[i] for i in indices[0]]

    print("\n==============================")
    print("CLAIM:", claim)
    print("Similarity:", round(similarity, 3))

    # ✅ GOOD MATCH
    if similarity >= threshold:
        print("Using FAISS corpus")
        return local_results

    # 🌐 FALLBACK
    print("Using WEB fallback")
    
    web_results = web_search_serpapi(claim)

    web_results = clean_web_results(web_results)

    print("Web results:", len(web_results))

    add_to_corpus(web_results)

    return web_results if web_results else local_results

In [48]:
def verify_pipeline(claim):

    evidences = hybrid_retrieve(claim, use_mock=False)

    verification_results = verify_with_evidence(claim, evidences)

    decision = final_decision(verification_results)

    return {
        "claim": claim,
        "decision": decision,
        "details": verification_results
    }

In [49]:
claims = [
    "A fusion reactor in South Korea generated sustained energy for 48 hours in 2026",
    "India launched a nationwide AI surveillance system in 2026",
    "A solar flare disrupted global satellite communication networks in 2026"
]

In [50]:
for claim in claims:

    output = verify_pipeline(claim)

    print("\n==============================")
    print("CLAIM:", output["claim"])
    print("FINAL:", output["decision"])

    print("\nDetails:")
    for e, label, score in output["details"]:
        print(f"{label} ({score}) → {e}")


CLAIM: A fusion reactor in South Korea generated sustained energy for 48 hours in 2026
Similarity: 0.673
Using FAISS corpus

CLAIM: A fusion reactor in South Korea generated sustained energy for 48 hours in 2026
FINAL: NEUTRAL

Details:
neutral (0.999) → In 2026, South Korea will achieve a historic energy milestone in plasma fusion, and the research team will break records that they had ...
neutral (0.998) → South Korea's KSTAR reactor has achieved a new milestone in nuclear fusion by maintaining superheated plasma at 100 million°C for 48 seconds, ...
contradiction (0.646) → South Korea plans to begin nuclear fusion power generation tests as early as 2030, nearly 20 years ahead of schedule, citing rising electricity ...
neutral (0.995) → The experimental fusion reactor sustained temperatures of 180 million degrees Fahrenheit for a record-breaking 48 seconds.
neutral (0.987) → South Korean researchers have maintained plasma temperatures of 100 million degrees Celsius for 48 seconds ins